# 🏆 V3 Enterprise Smart Miner (Standalone Edition)


Phiên bản này kết hợp **sức mạnh của thuật toán V3 (K-NN Calibration, 2-Pass Scanning, Quota Selection)** vào **một file Notebook duy nhất**, không phụ thuộc vào bất kỳ thư mục `src/` nào.


## 1. Cài Đặt Thư Viện


In [ ]:
!pip install ultralytics opencv-python-headless scikit-learn imagehash scikit-image pandas matplotlib tqdm


## 2. Mount Drive & Cấu Hình


In [ ]:
from google.colab import drive
import os, cv2, imagehash
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from tqdm.notebook import tqdm
from PIL import Image

drive.mount('/content/drive')

# --- CẤU HÌNH ---
PROJECT_ROOT = '/content/drive/MyDrive/Bradford_Bulls_AI'
VIDEO_NAME = 'M06_black_1080p.mp4'

VIDEO_PATH = os.path.join(PROJECT_ROOT, 'videos', VIDEO_NAME)
OUT_DIR = os.path.join(PROJECT_ROOT, 'extracted_dataset', f"{VIDEO_NAME.split('.')[0]}_v3")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"📁 Thư mục gốc: {PROJECT_ROOT}")
print(f"📁 Nguồn: {VIDEO_PATH}")
print(f"📁 Lưu tại: {OUT_DIR}")



## 3. Các Hàm Lõi (Core Functions)


In [ ]:
# --- 3.1. OVERLAY DETECTION ---
def detect_static_overlays(video_path, num_samples=20):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    samples = []
    for i in np.linspace(total*0.05, total*0.95, num_samples, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret: samples.append(cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (0,0), fx=0.5, fy=0.5))
    cap.release()
    
    variance = np.var(np.stack(samples), axis=0)
    thresh = max(np.percentile(variance, 3), 8.0)
    static_mask = (variance < thresh).astype(np.uint8) * 255
    kernel = np.ones((11, 11), np.uint8)
    static_mask = cv2.morphologyEx(static_mask, cv2.MORPH_CLOSE, kernel)
    
    # Scale back
    cap = cv2.VideoCapture(video_path)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    full_mask = cv2.resize(static_mask, (w, h), interpolation=cv2.INTER_NEAREST)
    return full_mask

# --- 3.2. CALIBRATION (K-NN) ---
def extract_torso_hist(frame, box, overlay_mask):
    x1, y1, x2, y2 = map(int, box)
    bw, bh = x2 - x1, y2 - y1
    if bw < 30 or bh < 50: return None
    
    # Lấy vùng ngực áo (10% - 40% chiều cao)
    ty1, ty2 = y1 + int(bh * 0.1), y1 + int(bh * 0.4)
    tx1, tx2 = x1 + int(bw * 0.1), x2 - int(bw * 0.1)
    if ty2 <= ty1 or tx2 <= tx1: return None
    
    # Bỏ qua nếu vùng áo dính Overlay (chữ, logo đài)
    if overlay_mask[ty1:ty2, tx1:tx2].mean() > 100: return None
    
    torso = frame[ty1:ty2, tx1:tx2]
    hsv = cv2.cvtColor(torso, cv2.COLOR_BGR2HSV)
    
    # Loại bỏ màu cỏ xanh
    green_mask = cv2.inRange(hsv, (35, 40, 40), (85, 255, 255))
    valid_mask = (green_mask == 0).astype(np.uint8)
    
    hist = cv2.calcHist([hsv], [0, 1], valid_mask, [12, 5], [0, 180, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

def classify_player(hist, target_refs, opponent_refs):
    if hist is None: return "ambiguous"
    d_target = min([cv2.compareHist(hist, r, cv2.HISTCMP_BHATTACHARYYA) for r in target_refs]) if target_refs else 1.0
    d_opp = min([cv2.compareHist(hist, r, cv2.HISTCMP_BHATTACHARYYA) for r in opponent_refs]) if opponent_refs else 1.0
    
    if d_target < 0.4 and d_target < d_opp: return "target"
    if d_opp < 0.4 and d_opp < d_target: return "opponent"
    return "ambiguous"

# --- 3.3. SCORING & FILTERING ---
def compute_sharpness(frame, box):
    x1, y1, x2, y2 = map(int, box)
    crop = frame[max(0, y1):y2, max(0, x1):x2]
    if crop.size == 0: return 0
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()



## 4. Phase 0A & 0B: Overlay & Thu thập mẫu


In [ ]:
yolo = YOLO("yolov8n.pt")

print("🔍 Đang phân tích Overlay...")
OVERLAY_MASK = detect_static_overlays(VIDEO_PATH)

print("🏃 Đang thu thập mẫu cầu thủ...")
cap = cv2.VideoCapture(VIDEO_PATH)
sample_crops, sample_hists = [], []
frame_idx = 0
while cap.isOpened() and len(sample_crops) < 24:
    frame_idx += 90
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret: break
    
    res = yolo.predict(frame, classes=[0], conf=0.5, verbose=False)
    if res[0].boxes is not None:
        for box in res[0].boxes.xyxy.cpu().numpy():
            hist = extract_torso_hist(frame, box, OVERLAY_MASK)
            if hist is not None:
                x1, y1, x2, y2 = map(int, box)
                sample_crops.append(frame[y1:y2, x1:x2])
                sample_hists.append(hist)
                if len(sample_crops) >= 24: break
cap.release()

fig, axes = plt.subplots(4, 6, figsize=(15, 10))
for i, ax in enumerate(axes.flat):
    if i < len(sample_crops):
        ax.imshow(cv2.cvtColor(cv2.resize(sample_crops[i], (64, 128)), cv2.COLOR_BGR2RGB))
        ax.set_title(f"ID: {i}", color="red", fontweight="bold")
    ax.axis("off")
plt.show()


## 5. Chọn Đội (K-NN References)


In [ ]:
raw_target = input("👉 Nhập các ID của đội Bradford Bulls (ví dụ: 0, 3, 5): ")
TARGET_IDS = [int(x.strip()) for x in raw_target.split(",") if x.strip().isdigit()]

TARGET_REFS = [sample_hists[i] for i in TARGET_IDS]
OPPONENT_REFS = [sample_hists[i] for i in range(len(sample_hists)) if i not in TARGET_IDS]
print(f"✅ Đã lưu {len(TARGET_REFS)} mẫu Home và {len(OPPONENT_REFS)} mẫu Away.")


## 6. Pass 1: Fast Scan (Tìm phân đoạn chất lượng)


In [ ]:
print("🚀 Bắt đầu Pass 1: Fast Scan...")
cap = cv2.VideoCapture(VIDEO_PATH)
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FRAME_AREA = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) * cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

quality_segments = []
pbar = tqdm(total=TOTAL)

frame_idx = 0
while cap.isOpened():
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret: break
    
    res = yolo.predict(frame, classes=[0], conf=0.3, verbose=False)
    max_area_ratio = 0
    if res[0].boxes is not None:
        for box in res[0].boxes.xyxy.cpu().numpy():
            area = (box[2]-box[0]) * (box[3]-box[1])
            max_area_ratio = max(max_area_ratio, area / FRAME_AREA)
            
    # Zoom-in camera (Quality segment)
    if max_area_ratio > 0.015:
        quality_segments.append(frame_idx)
        
    frame_idx += 5
    pbar.update(5)

cap.release()
pbar.close()
print(f"✅ Pass 1: Tìm thấy {len(quality_segments)} frames tiềm năng.")


## 7. Pass 2: Team-Aware Extraction & Scoring


In [ ]:
print("🔬 Bắt đầu Pass 2: Chiết xuất và Chấm điểm...")
cap = cv2.VideoCapture(VIDEO_PATH)
candidates = []

for fn in tqdm(quality_segments):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fn)
    ret, frame = cap.read()
    if not ret: continue
    
    res = yolo.predict(frame, classes=[0], conf=0.35, verbose=False)
    if res[0].boxes is None: continue
    
    n_target, n_opp = 0, 0
    max_sharpness = 0
    max_person_ratio = 0
    
    for box in res[0].boxes.xyxy.cpu().numpy():
        area = (box[2]-box[0]) * (box[3]-box[1])
        max_person_ratio = max(max_person_ratio, area / FRAME_AREA)
        
        hist = extract_torso_hist(frame, box, OVERLAY_MASK)
        team = classify_player(hist, TARGET_REFS, OPPONENT_REFS)
        if team == "target": n_target += 1
        elif team == "opponent": n_opp += 1
        
        sharpness = compute_sharpness(frame, box)
        max_sharpness = max(max_sharpness, sharpness)

    # Bỏ qua frames quá mờ (Motion Blur cứng)
    if max_sharpness < 20.0: continue
    
    # Tính điểm toàn diện
    n_total = max(n_target + n_opp, 1)
    score = (max_sharpness/1000) * 0.3 + (max_person_ratio*10) * 0.3 + (n_target/n_total) * 0.4
    
    cat = "target" if n_target > n_opp else ("mixed" if n_target > 0 else "opponent")
    
    candidates.append({
        "frame_num": fn,
        "score": score,
        "category": cat,
        "frame": frame.copy() # Lưu frame trực tiếp vào RAM (nếu RAM đủ lớn)
    })
cap.release()
print(f"✅ Pass 2: Đã chấm điểm {len(candidates)} frames hợp lệ.")


## 8. Quota Selection & Lưu Kết Quả


In [ ]:
QUOTA = {"target": 0.60, "mixed": 0.30, "opponent": 0.10}
TARGET_TOTAL_FRAMES = 500

candidates.sort(key=lambda x: x["score"], reverse=True)
selected = []
counts = {"target": 0, "mixed": 0, "opponent": 0}

for c in candidates:
    cat = c["category"]
    if counts[cat] < TARGET_TOTAL_FRAMES * QUOTA[cat]:
        # Deduplication bằng pHash đơn giản
        curr_hash = imagehash.phash(Image.fromarray(cv2.cvtColor(c["frame"], cv2.COLOR_BGR2RGB)))
        is_dup = False
        for s in selected:
            if abs(curr_hash - s["hash"]) < 10:
                is_dup = True
                break
        
        if not is_dup:
            selected.append({"frame_num": c["frame_num"], "frame": c["frame"], "hash": curr_hash})
            counts[cat] += 1
            
        if len(selected) >= TARGET_TOTAL_FRAMES: break

print("💾 Đang lưu file vào Drive...")
for s in tqdm(selected):
    fn = s["frame_num"]
    ts = fn / 30.0
    h, m, sec = int(ts//3600), int((ts%3600)//60), int(ts%60)
    filename = f"V3_{fn:06d}_{h:02d}h{m:02d}m{sec:02d}s.jpg"
    cv2.imwrite(os.path.join(OUT_DIR, filename), s["frame"])

print(f"🎉 HOÀN TẤT! Đã lưu {len(selected)} frames cực phẩm vào {OUT_DIR}")
print(f"📊 Thống kê: {counts}")
